# পাঠ ১১ - মডেল কনটেক্সট প্রোটোকল (MCP)

**মডেল কনটেক্সট প্রোটোকল (MCP)** একটি ওপেন স্ট্যান্ডার্ড যা এজেন্টদের রানটাইমে টুল, সম্পদ এবং ডেটা সোর্সগুলি গতিশীলভাবে আবিষ্কার এবং ব্যবহার করতে সক্ষম করে। একটি এজেন্টে সরাসরি টুলগুলো হার্ডকোড করার পরিবর্তে, MCP এজেন্টদের এমন বাহ্যিক সার্ভারের সাথে সংযোগ স্থাপন করার সুযোগ দেয় যা প্রয়োজন অনুযায়ী ক্ষমতাগুলো প্রকাশ করে।

এই পাঠে, আপনি শিখবেন:
- MCP কি এবং কেন এটি এজেন্ট সিস্টেমের জন্য গুরুত্বপূর্ণ
- MCP ক্লায়েন্ট-সার্ভার স্থাপত্য কিভাবে কাজ করে
- কীভাবে MCP-স্টাইলে টুল আবিষ্কারের জন্য এজেন্ট তৈরি করবেন


## সেটআপ

**অগ্রশর্তসমূহ:**
- একটি Microsoft Foundry প্রকল্প যার একটি মোডেল মোতায়েন করা হয়েছে
- প্রমাণীকরণের জন্য `az login` চালান


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## মডেল কনটেক্সট প্রটোকল (MCP) কী?

MCP AI এজেন্টদের জন্য একটি স্ট্যান্ডার্ড উপায় নির্ধারণ করে যাতে তারা বাহ্যিক সরঞ্জাম এবং ডেটা উৎস আবিষ্কার এবং ইন্টারঅ্যাক্ট করতে পারে:

- **MCP সার্ভার**: একটি স্ট্যান্ডার্ড প্রটোকল মাধ্যমে সরঞ্জাম, সম্পদ এবং প্রোম্পট এক্সপোজ করে
- **MCP ক্লায়েন্ট**: সেই এজেন্ট রানটাইম যা সার্ভারগুলোর সাথে সংযুক্ত হয় এবং উপলব্ধ সক্ষমতাগুলো আবিষ্কার করে
- **ডায়নামিক ডিসকভারি**: এজেন্টদের হার্ডকোডেড সরঞ্জামের প্রয়োজন নেই — তারা রানটাইমে যা পাওয়া যায় তা আবিষ্কার করে

এটি একটি শক্তিশালী পদ্ধতি এক্সটেনসিবল এজেন্ট সিস্টেম তৈরি করার জন্য যেখানে নতুন সক্ষমতা যুক্ত করা যায় এজেন্ট কোড পরিবর্তন ছাড়াই।


## এমসিপি কীভাবে কাজ করে

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. এজেন্ট (MCP ক্লায়েন্ট) একটি MCP সার্ভারের সঙ্গে সংযোগ স্থাপন করে
2. সার্ভার উপলব্ধ টুল এবং তাদের স্কিমাগুলির একটি তালিকা দিয়ে প্রতিক্রিয়া জানায়
3. এজেন্ট তার যুক্তি চলাকালীন যে কোনো আবিষ্কৃত টুল কল করতে পারে
4. ফলাফল একই প্রোটোকলের মাধ্যমে ফিরিয়ে আনা হয়


## এমসিপি টুল আবিষ্কারের সিমুলেশন

যেহেতু একটি প্রকৃত এমসিপি সার্ভারের জন্য চালু থাকা সার্ভার প্রসেস প্রয়োজন, আমরা `@tool` ফাংশন ব্যবহার করে এমন একটি প্যাটার্ন প্রদর্শন করব যা এমসিপি-সংযুক্ত অ্যাকোমোডেশান সার্ভিস যা প্রদান করবে তা সিমুলেট করে।

প্রোডাকশনে, এই টুলগুলো লোকালি সংজ্ঞায়িত করার পরিবর্তে ডাইনামিকভাবে একটি এমসিপি সার্ভার থেকে আবিষ্কৃত হবে।


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-শৈলীর টুলস দিয়ে একটি এজেন্ট নির্মাণ


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## প্রোডাকশন পরিবেশে MCP

একটি প্রোডাকশন পরিবেশে, MCP শক্তিশালী প্যাটার্নগুলি সক্ষম করে:

- **ডায়নামিক টুল আবিষ্কার**: এজেন্টরা MCP সার্ভারের সাথে সংযোগ স্থাপন করে এবং রানটাইমে টুল আবিষ্কার করে
- **বিচ্ছিন্ন স্থাপত্য**: টুল প্রদানকারীরা এজেন্ট থেকে স্বাধীনভাবে আপডেট করতে পারে
- **প্রতিষ্ঠানের সীমানা অতিক্রম করে শেয়ারিং**: দলগুলি MCP সার্ভারের মাধ্যমে ক্ষমতা প্রদর্শন করতে পারে যা যে কোনও এজেন্ট ব্যবহার করতে পারে
- **মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক সমর্থন**: MAF-এর মধ্যে `mcp` ইন্টিগ্রেশনের মাধ্যমে নেটিভ MCP ক্লায়েন্ট সমর্থন রয়েছে

MAF-এর সাথে একটি বাস্তব MCP সার্ভার ব্যবহার করতে, আপনি `hosted_mcp_tool()` অথবা MCP ক্লায়েন্ট ইন্টিগ্রেশনের মাধ্যমে সংযোগ করবেন।

**আরও জানুন:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Summary

In this lesson, you learned:
- **MCP** is an open standard for dynamic tool discovery between agents and tool providers
- The **client-server architecture** lets agents discover capabilities at runtime
- MCP enables **extensible, decoupled agent systems** where tools can be added without code changes
- Microsoft Agent Framework provides **built-in MCP support** for production use


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
